# 📘 Introduction to Machine Learning with Scikit-Learn

**AIRCHECK Workshop 2026** · From data preparation to model evaluation — the core ML workflow in Python.

Welcome to this **introductory guide to Machine Learning (ML)**! This notebook covers essential ML
concepts using **Scikit-Learn**, starting from data preparation to model evaluation.

---

# 🔍 Section 1 · What is Machine Learning?

Machine Learning (ML) is a field of computer science that focuses on finding patterns in data. In hyper-simplified non-functional code, the procedure is:

In [ ]:
# patterns = ml_algorithm(data)

There are two popular kinds of problems, also called learning scenarios, that ML practitioners confront.

The simplest learning scenario is called **Unsupervised Learning**. In this case we want to find some hidden structure in the data provided. An archetypal problem in unsupervised learning is **clustering**. Clustering involves taking in a collection $X$ of $n$ vectors/arrays of real numbers $\{x_1, \dots, x_n\}$ where $x_i\in R^m$ and partitioning them into disjoint groups.

In [ ]:
#clusters = clustering_algorithm(X)

The other learning scenario we consider is **supervised learning**. In this case each of the *observation* $x_i$ has a matched **label** $y_i$. The assumption is that there is some function $f$ that connects the observations to the labels, that is
$y_i = f(x_i)$.

The goal of ML in supervised learning is to *learn a model*  $m_f$ that “behaves like $f$.” That is we want $m_f(x_i)$ to be close to $y_i$ for all $i$. Once $m_f$ has been constructed, we can use it to predict the labels on new data $X'$ where we may not know the labels. Again in cartoon code:

In [ ]:
#model = fit_model(X,y)
#predicted_labels = model.predict(X_new)

Supervised learning can be further subdivided into **regression problems** and **classification problems** based on the nature of their labels.

Regression problems have continuous/real-valued labels. An example regression problem is predicting the price of a house from various measurements about it (e.g. size, neighborhood, number of bedrooms, etc.).

For classification problems the labels come from a finite set of categories. An example classification problem is categorizing an e-mail as spam or not spam.

Fortunately we do not need to rewrite ML algorithms from scratch. In python there is a comprehensive library called `scikit-learn` that has implemented many popular and effective machine learning algorithms that can be used "off the shelf."  We will use `scikit-learn` to illustrate the basics of machine learning.

---

# 📂 Section 2 · Loading & Exploring a Dataset

We are still missing a key ingredient: a dataset. For this we can use one of the many datasets included with `scikit-learn`. In particular we will use the commonly analyzed **Iris** dataset. This dataset contains information about three species of iris flowers. We will, over this and the following sections, build a classifier that inputs information and can classifies them as one of the three species of iris: *setosa*, *versicolor*, and *virginica*.

In addition to `scikit-learn`, which focuses primarily on the machine learning side of things, we need two addition libraries: **NumPy** and **Pandas**. These libraries are used to manage and process data before and after analyzing it with `scikit-learn`. These libraries are installed in the normal way:

In [ ]:
import numpy as np
import pandas as pd


Now we load the dataset proper.

In [ ]:
from sklearn.datasets import load_iris
iris = load_iris()

As written the `iris` object is a dictionary with our data and the labels (also called targets) as well as some additional information or *metadata*. As a first step let's pull out the observations $X$ and the labels $y$:

In [ ]:
X, y = iris['data'],iris['target']

Okay, now what do these look like:

In [ ]:
X

In [ ]:
y

This is... not particularly useful. Let's try finding out how large the datasets are. To do this we will use the `.shape` attribute of `NumPy` arrays.

In [ ]:
print(X.shape)
print(y.shape)

So there are 150 observations in our data and $X$ has 4 *features* per observation. It would be good to know what measure each feature (column) of $X$ maps to. We can do this with a simple call.

In [ ]:
iris['feature_names']

It's clear from the code above that $y$ has three categories, encoded as numbers 0, 1, and 2. Finding out which species of iris these correspond to is doable with another line.

In [ ]:
iris['target_names']

It's worth  pulling together a count of how many observations we have for each category.

In [ ]:
label_to_species = {i: iris["target_names"][i] for i in range(len(iris["target_names"]))}
renamed_targets = [label_to_species[j] for j in y]
pd.Series(renamed_targets).value_counts()

As a last step we want to look at features. To do this we make a pandas dataframe (which you can think of as a spreadsheet with some extra bells and whistles) and analyse their properties.

In [ ]:
data = pd.DataFrame(data=X, columns=iris['feature_names'])
print("Mean Value of Each Feature")
print(data.mean(axis=0))
print("\nStandard Deviation of Each Feature")
print(data.std(axis=0))
print("\nCorrelation Between Features")
print(data.corr())

---

# 📐 Section 3 · Splitting Data into Train & Test Sets
As mentioned above supervised learning is designed to predict the response (species) on *new* data. However, we only have the one dataset! In order to evaluate an algorithm on this dataset we randomly divide it into two subsets: a **training** set and a **testing** set. This partition of $X$ is called a *train-test-split*. Usually the training set is much larger than the test set. We set it to be 80% of the data in this example.

`scikit-learn` has built in functionality to perform train-test splits. In order to keep things reproducible we set a random seed so that the train-test split is consistent.

In [ ]:
from sklearn.model_selection import train_test_split

#set the seed
seed = 123456
train_percentage = 0.8
X_train, X_test, y_train, y_test = train_test_split(X,y,train_size = train_percentage, random_state=seed)
print(f"The training data has {X_train.shape[0]} observations")
print(f"The testing data has {X_test.shape[0]} observations")

We should probably also check how many of each species are in the training and testing sets...

In [ ]:
training_names = [label_to_species[j] for j in y_train]
testing_names = [label_to_species[j] for j in y_test]

print(f"The counts of species in the training dataset is:\n-----------\n{pd.Series(training_names).value_counts()}\n")
print(f"The counts of species in the testing dataset is:\n-----------\n{pd.Series(testing_names).value_counts()}")

We started with an exactly equal number of each species in the dataset but we have changed this (slightly) by splitting it. This is something to note as we train our model.

---

# ⚙️ Section 4 · Training a Machine Learning Model
Now that we've split our data it's time to train our model. We'll use a simple linear model called **logistic regression** (see the [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) for details).

As the names suggest we will use our training data to fit (train) the model and then pass the testing data through it for evaluation.

In [ ]:
from sklearn.linear_model import LogisticRegression
classifier = LogisticRegression()
classifier.fit(X_train,y_train)
predictions = classifier.predict(X_test)
predictions

It is good (and common) practice to *standardize* the data by make it so each feature has mean zero and unit variance. We can use the `StandardScaler` functionality `scikit-learn` has.

In [ ]:
from sklearn.preprocessing import StandardScaler
standardizer = StandardScaler()
# note that the scaler does NOT care about the targets.
standardizer.fit(X_train)


Does standardizing help? Do we do better if we preserve the class distribution when we split the train and test data? In order to answer these questions we need a way to evaluate model performance.

---

# 📊 Section 5 · Evaluating Model Performance
- Using **accuracy, precision, recall, and F1-score** for classification.
- Using **Mean Squared Error (MSE) and R² score** for regression.
- Displaying results using `classification_report()` and `confusion_matrix()`.

In [ ]:
from sklearn.metrics import classification_report, accuracy_score
# print(classification_report(y_test, predictions))
print(f"Accuracy is {100*accuracy_score(y_test,predictions)}%")

How nice! We got 100% accuracy. This, however, doesn't tell us very much. Let's try this again but with a smaller percentage of the data used for training. We check across a range of values.

In [ ]:
train_percentages = np.round((10**-1)*np.arange(3,10,0.5),2)
# increase the total number of iterations used to fit the model
classifier = LogisticRegression(max_iter=1000)
for tpct in train_percentages:
  X_train, X_test, y_train, y_test = train_test_split(X,y,train_size = tpct, random_state=seed)
  classifier.fit(X_train,y_train)
  predictions = classifier.predict(X_test)
  print(f"Train Percentage: {np.round(100*tpct,2)} % ---- Accuracy:{np.round(100*accuracy_score(y_test,predictions),2)}%")


So, as one would expect as we increase the amount of data used for model training the better the model does. However, even at low training data we do quite well. Let's try this again but this time with the class distributions preserved. We will use this with the `stratify` argument.

In [ ]:
print("For stratified splits...")
for tpct in train_percentages:
  X_train, X_test, y_train, y_test = train_test_split(X,y,train_size = tpct, random_state=seed,stratify=y)
  classifier.fit(X_train,y_train)
  predictions = classifier.predict(X_test)
  print(f"Train Percentage: {np.round(100*tpct,2)} % ---- Accuracy:{np.round(100*accuracy_score(y_test,predictions),2)}%")


Interesting! If we force the class distribution to be preserved then the model seems to struggle (more). We also see that the relationship between amount of training data and performance is less consistent. What if we try standardizing our data?

In [ ]:
print("For stratified and standardized splits...")
for tpct in train_percentages:
  X_train, X_test, y_train, y_test = train_test_split(X,y,train_size = tpct, random_state=seed,stratify=y)
  scaler = StandardScaler()
  scaler.fit(X_train)
  X_train = scaler.transform(X_train)
  X_test = scaler.transform(X_test)
  classifier.fit(X_train,y_train)
  predictions = classifier.predict(X_test)
  print(f"Train Percentage: {np.round(100*tpct,2)} % ---- Accuracy:{np.round(100*accuracy_score(y_test,predictions),2)}%")


So standardizing doesn't do much. However, this is likely due to the fact that the features are all measured in the same units (cm) and are roughly of the same scale. However, this is not usually true of the datasets we see in practice. In general:

- Your **default** move should be to standardize your data.
- Standardization **must** be done on the train set. If you fit `StandardScaler` on the full data then information will leak and you will over-estimate model performance.

---

# 🎯 Section 6 · Hyperparameter Tuning
If you look over the logistic regression documentation you'll see two parameters; a "penalty" parameter and a parameter "C". We are going to try to choose these values (called **hyperparameters**) in order to maximize model performance. In order to do this we will avail ourselves of two useful features that `scikit-learn` has: `Pipeline` and `GridSearchCV`. We will use `Pipeline` to wrap our train-test split protocol and `GridSearchCV` to try all possible combinations of hyperparameters.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

# The Pipeline is a named list of steps that get applied in order.
# Since we will be trying multiple penalties we need to specify a
# solver that can handle both.
model_pipe = Pipeline([('scale',StandardScaler()),('clf',LogisticRegression(solver='liblinear',max_iter=1000))])

# Now we specify the parameters to search over.
# This takes a form of a dictionary where the parameter names are
# taken from the document and we add a prefix so that
# GridSearchCV knows to which step the parameters belong.
# These need to come after two underscores.

param_grid = {
    'clf__C':np.logspace(-4, 4, 100),
    'clf__penalty':['l1','l2']}

In order to choose hyperparameters we first split the data then use what is called stratified k-fold cross validation. In this process the training data is split into k equal subsets, the model is then fit to the remaining k-1 subsets (folds) and evaluated on the held-out fold. The model parameters with the best average performance over all k folds are kept as the best. We choose $k=5$ because our dataset is small. `scikit-learn` uses stratified kfold splits by default for classification problems.

In [ ]:
model = GridSearchCV(model_pipe,param_grid,cv=5)
print("For stratified and standardized splits...")
for tpct in train_percentages:
  X_train, X_test, y_train, y_test = train_test_split(X,y,train_size = tpct, random_state=seed,stratify=y)
  model.fit(X_train,y_train)

  predictions = model.predict(X_test)
  print(f"Train Percentage: {np.round(100*tpct,2)} % ---- Accuracy:{np.round(100*accuracy_score(y_test,predictions),2)}%")



We see that tuning the hyperparameters increases our performance, albeit modestly. Once we have fit the model we can use the best hyper-parameters saved to re-fit on the same model. Let's re-do this with 60% of the data held out. The `GridSearchCV` object has an attribute called `best_estimator_` that stores the model with the best hyperparameters.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,train_size = 0.6, random_state=seed,stratify=y)
model.fit(X_train,y_train)
print(model.best_estimator_)



---

# 💾 Section 7 · Saving & Loading a Trained Model
Once we've built and trained our model we want to save it for future use so that we (or others) won't need to re-train it. There are many [options](https://scikit-learn.org/stable/model_persistence.html) for this in python. We'll use `joblib` for this example.

In [ ]:
import joblib

best_model = model.best_estimator_
joblib.dump(best_model,"./my_model.joblib")
loaded_model = joblib.load("./my_model.joblib")
loaded_model.predict(X_test)

Alright! The model works and can be loaded. The last step, if we want to apply to genuinely new data, is to re-fit the best model on all our data and save it.

In [ ]:
production_model = model.best_estimator_
production_model.fit(X,y)
joblib.dump(production_model,"./production_model.joblib")